In [ ]:
import pandas as pd
import numpy as np
from itertools import product, combinations


# 데이터 로드
df = pd.read_csv(r'C:\Users\3eunh\OneDrive\Desktop\NS\code\data\05_train_with_category.csv')

print("="*70)
print("카테고리별 샘플 수")
print("="*70)
print(df['Category'].value_counts())

# Rule 설정
rule_config = {
    'Brute Force': {'Bwd Pkts/s': 'high', 'Init Fwd Win Byts': 'high'},
    'Botnet': {'Flow Duration': 'low', 'Pkt Len Mean': 'low'},
    'DoS/DDoS': {'PSH Flag Cnt': 'high', 'Init Fwd Win Byts': 'high'},
    'Web Attack': {'Subflow Fwd Byts': 'high', 'Fwd Header Len': 'high', 'Fwd Pkt Len Std': 'high'}
}

# 카테고리별 FPR 제한 (α)
fpr_limits = {
    'Brute Force': 0.01,   # 엄격 (이미 좋음)
    'Botnet': 0.05,        # 완화 (현재 21% → 5%로)
    'DoS/DDoS': 0.05,      # 완화 (현재 21% → 5%로)
    'Web Attack': 0.01     # 엄격 (이미 좋음)
}

# 실험별 설정
experiment_config = {
    'Web Attack': {
        'mode': 'k-of-n',
        'k': 2,  # 3개 중 2개 만족
        'alpha': 0.01
    },
    'DoS/DDoS': {
        'mode': 'alpha_sweep',
        'alpha': 0.10  # 5% → 10%로 완화
    },
    'Brute Force': {
        'mode': 'standard',
        'alpha': 0.01
    },
    'Botnet': {
        'mode': 'skip',  # 실험 제외 (발표용 실패 케이스)
        'alpha': 0.05
    }
}

# 후보 quantile 확장 (촘촘하게)
quantiles_high = [0.90, 0.925, 0.95, 0.975, 0.99, 0.995, 0.999]  # benign 기준 (direction=high)
quantiles_low = [0.10, 0.075, 0.05, 0.025, 0.01, 0.005, 0.001]   # benign 기준 (direction=low)
attack_quantiles_high = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25]     # attack 기준 (direction=high)
attack_quantiles_low = [0.99, 0.95, 0.90, 0.85, 0.80, 0.75]      # attack 기준 (direction=low)


카테고리별 샘플 수
Category
Benign         11307
DoS/DDoS        8145
Brute Force     2324
Botnet          1327
Web Attack       743
Name: count, dtype: int64


In [33]:
# Step 1: >= / <= 로 변경 + FPR 반환 추가
# ============================================================================
def clean_feature_series(s, feature: str):
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if feature in ["Init Fwd Win Byts"]:
        s = s[s >= 0]
    return s

def get_threshold_candidates(df, category, feature, direction):
    """
    확장된 threshold 후보 생성
    """
    benign = clean_feature_series(df[df['Category'] == 'Benign'][feature], feature)
    attack = clean_feature_series(df[df['Category'] == category][feature], feature)
    
    candidates = {}
    
    if direction == 'high':
        # Benign 상위 quantile
        for q in quantiles_high:
            val = benign.quantile(q)
            candidates[f'benign_p{int(q*100)}'] = val
        # Attack 하위 quantile
        for q in attack_quantiles_high:
            val = attack.quantile(q)
            candidates[f'attack_p{int(q*100)}'] = val
    else:  # low
        # Benign 하위 quantile
        for q in quantiles_low:
            val = benign.quantile(q)
            candidates[f'benign_p{int(q*100)}'] = val
        # Attack 상위 quantile
        for q in attack_quantiles_low:
            val = attack.quantile(q)
            candidates[f'attack_p{int(q*100)}'] = val
    
    # NaN 제거 + 중복 제거
    cleaned = {}
    seen_vals = set()
    for name, val in candidates.items():
        if val is None or (isinstance(val, float) and np.isnan(val)):
            continue
        key = round(float(val), 6)
        if key in seen_vals:
            continue
        seen_vals.add(key)
        cleaned[name] = float(val)
    
    return cleaned

def check_single_condition(val, threshold, direction):
    """단일 조건 체크"""
    if pd.isna(val) or np.isinf(val):
        return False
    if direction == 'high':
        return val >= threshold
    else:
        return val <= threshold

def evaluate_and_rule(df, category, thresholds_dict):
    """
    AND rule 전체를 하나의 classifier로 평가 (One-vs-Benign)
    """
    attack_df = df[df['Category'] == category].copy()
    benign_df = df[df['Category'] == 'Benign'].copy()
    
    def check_and_rule(row):
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            if pd.isna(val) or np.isinf(val):
                return False
            if feature == "Init Fwd Win Byts" and val < 0:
                return False
            
            if direction == 'high':
                if val < threshold:
                    return False
            else:
                if val > threshold:
                    return False
        return True
    
    attack_hits = attack_df.apply(check_and_rule, axis=1)
    benign_hits = benign_df.apply(check_and_rule, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr
    }
    
def check_single_condition(val, threshold, direction):
    """단일 조건 체크"""
    if pd.isna(val) or np.isinf(val):
        return False
    if direction == 'high':
        return val >= threshold
    else:
        return val <= threshold


def evaluate_k_of_n_rule(df, category, thresholds_dict, k):
    """k-of-n rule 평가: n개 조건 중 k개 이상 만족하면 탐지"""
    attack_df = df[df['Category'] == category]
    benign_df = df[df['Category'] == 'Benign']
    
    def check_k_of_n(row):
        satisfied = 0
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            if feature == "Init Fwd Win Byts" and val < 0:
                continue
            if check_single_condition(val, threshold, direction):
                satisfied += 1
        return satisfied >= k
    
    attack_hits = attack_df.apply(check_k_of_n, axis=1)
    benign_hits = benign_df.apply(check_k_of_n, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'Precision': precision, 'Recall': recall, 'F1': f1, 'FPR': fpr
    }


In [34]:
# Threshold 분석
# ============================================================================
all_results = []

for category, features in rule_config.items():
    print(f"\n{'='*70}")
    print(f"[{category}]")
    print("="*70)
    
    for feature, direction in features.items():
        benign = clean_feature_series(df[df['Category'] == 'Benign'][feature], feature)
        attack = clean_feature_series(df[df['Category'] == category][feature], feature)
        
        print(f"\n--- {feature} ({direction}) ---")
        print(f"Benign: mean={benign.mean():.4f}, median={benign.median():.4f}")
        print(f"Attack: mean={attack.mean():.4f}, median={attack.median():.4f}")
        
        # Threshold 후보
        if direction == 'high':
            candidates = {
                'benign_p95': benign.quantile(0.95),
                'benign_p99': benign.quantile(0.99),
                'attack_p5': attack.quantile(0.05),
                'attack_p25': attack.quantile(0.25)
            }
        else:
            candidates = {
                'benign_p5': benign.quantile(0.05),
                'benign_p1': benign.quantile(0.01),
                'attack_p95': attack.quantile(0.95),
                'attack_p75': attack.quantile(0.75)
            }
        
        # 후보값 정리: NaN 제거 + 중복 제거
        cleaned = {}
        seen_vals = set()

        for name, val in candidates.items():
            if val is None or (isinstance(val, float) and np.isnan(val)):
                continue

            key = round(float(val), 6)
            if key in seen_vals:
                continue

            seen_vals.add(key)
            cleaned[name] = float(val)

        candidates = cleaned

        
        # FPR 컬럼 추가
        print(f"\n{'Threshold':<15} {'Value':<15} {'Precision':<10} {'Recall':<10} {'F1':<10} {'FPR':<10}")
        print("-"*70)
        
        best_f1, best_name, best_value = 0, None, None
        for name, value in candidates.items():
            result = evaluate_threshold(df, category, feature, value, direction)
            print(f"{name:<15} {value:<15.4f} {result['Precision']:<10.4f} {result['Recall']:<10.4f} {result['F1']:<10.4f} {result['FPR']:<10.4f}")
            if result['F1'] > best_f1:
                best_f1, best_name, best_value, best_result = result['F1'], name, value, result
        
        print(f"\n★ 추천: {best_name} = {best_value:.4f} (F1={best_f1:.4f}, FPR={best_result['FPR']:.4f})")
        all_results.append({
            'Category': category, 'Feature': feature, 'Direction': direction,
            'Threshold_Name': best_name, 'Threshold': best_value,
            'Precision': best_result['Precision'], 'Recall': best_result['Recall'],
            'F1': best_f1, 'FPR': best_result['FPR']
        })

# 결과 DataFrame
results_df = pd.DataFrame(all_results)



[Brute Force]

--- Bwd Pkts/s (high) ---
Benign: mean=4640.9303, median=0.0312
Attack: mean=598121.2635, median=500000.0000

Threshold       Value           Precision  Recall     F1         FPR       
----------------------------------------------------------------------
benign_p95      43478.2609      0.8012     0.9849     0.8836     0.0502    
benign_p99      55555.5556      0.9258     0.9337     0.9297     0.0154    
attack_p5       52631.5789      0.8881     0.9660     0.9254     0.0250    
attack_p25      500000.0000     0.9908     0.7844     0.8756     0.0015    

★ 추천: benign_p99 = 55555.5556 (F1=0.9297, FPR=0.0154)

--- Init Fwd Win Byts (high) ---
Benign: mean=8277.8687, median=8192.0000
Attack: mean=26883.0000, median=26883.0000

Threshold       Value           Precision  Recall     F1         FPR       
----------------------------------------------------------------------
benign_p95      32818.1000      0.0000     0.0000     0.0000     0.0501    
benign_p99      65535.0000

In [35]:
# 최종 Rule 출력
print("최종 Threshold 요약")
print("="*70)
print(results_df[['Category', 'Feature', 'Direction', 'Threshold', 'Precision', 'Recall', 'F1', 'FPR']].to_string(index=False))

print("\n\n최종 Rule")
print("="*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    print(f"\n[{category}]")
    print("IF")
    for i, (_, row) in enumerate(cat_df.iterrows()):
        op = ">=" if row['Direction'] == 'high' else "<="
        print(f"    {row['Feature']} {op} {row['Threshold']:.4f}")
        if i < len(cat_df) - 1:
            print("  AND")
    print(f"THEN \"{category}\"")

최종 Threshold 요약
   Category           Feature Direction    Threshold  Precision   Recall       F1      FPR
Brute Force        Bwd Pkts/s      high 55555.555556   0.925768 0.933735 0.929734 0.015389
Brute Force Init Fwd Win Byts      high 26883.000000   0.818887 1.000000 0.900426 0.075555
     Botnet     Flow Duration       low 11204.000000   0.322829 0.949510 0.481836 0.233749
     Botnet      Pkt Len Mean       low    56.875000   0.169719 0.998493 0.290125 0.573273
   DoS/DDoS      PSH Flag Cnt      high     1.000000   0.615257 0.863475 0.718533 0.388963
   DoS/DDoS Init Fwd Win Byts      high  8192.000000   0.644934 1.000000 0.784145 0.569161
 Web Attack  Subflow Fwd Byts      high  3847.700000   0.629870 0.261104 0.369172 0.010082
 Web Attack    Fwd Header Len      high   811.280000   0.625000 0.255720 0.362942 0.010082
 Web Attack   Fwd Pkt Len Std      high   235.615693   0.365471 0.438762 0.398777 0.050057


최종 Rule

[Brute Force]
IF
    Bwd Pkts/s >= 55555.5556
  AND
    Init Fw

## “최종 AND rule 성능 평가” 함수부터 붙여서, 현재 threshold로 시스템 성능표가 바로 나오게 만들기



In [36]:
# Step 2: AND Rule 성능 평가 (One-vs-Benign)
# ============================================================================
print("\n\n" + "="*70)
print("Step 2: AND Rule 조합 성능 평가 (One-vs-Benign)")
print("="*70)

def evaluate_and_rule(df, category, thresholds_dict):
    """
    AND rule 전체를 하나의 classifier로 평가
    
    Parameters:
    - df: 전체 데이터
    - category: 평가할 공격 카테고리
    - thresholds_dict: {feature: (threshold, direction), ...}
    
    Returns:
    - 성능 지표 딕셔너리
    
    평가 기준:
    - TP: 해당 카테고리 샘플 중 rule hit
    - FN: 해당 카테고리 샘플 중 rule miss
    - FP: Benign 샘플 중 rule hit (오탐)
    - TN: Benign 샘플 중 rule miss
    """
    # 해당 카테고리와 Benign만 추출
    attack_df = df[df['Category'] == category].copy()
    benign_df = df[df['Category'] == 'Benign'].copy()
    
    # AND 조건 평가 함수
    def check_and_rule(row):
        for feature, (threshold, direction) in thresholds_dict.items():
            val = row[feature]
            # NaN이나 Inf 처리
            if pd.isna(val) or np.isinf(val):
                return False
            # Init Fwd Win Byts 음수 처리
            if feature == "Init Fwd Win Byts" and val < 0:
                return False
            
            if direction == 'high':
                if val < threshold:  # >= threshold를 만족 못하면 False
                    return False
            else:  # low
                if val > threshold:  # <= threshold를 만족 못하면 False
                    return False
        return True  # 모든 조건 만족
    
    # 각 샘플에 대해 rule 평가
    attack_hits = attack_df.apply(check_and_rule, axis=1)
    benign_hits = benign_df.apply(check_and_rule, axis=1)
    
    tp = attack_hits.sum()
    fn = len(attack_df) - tp
    fp = benign_hits.sum()
    tn = len(benign_df) - fp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'Category': category,
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'FPR': fpr,
        'Attack_Total': len(attack_df),
        'Benign_Total': len(benign_df)
    }


# results_df에서 현재 threshold 추출하여 AND rule 평가
and_rule_results = []

for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    
    # threshold dict 구성: {feature: (threshold, direction)}
    thresholds_dict = {}
    for _, row in cat_df.iterrows():
        thresholds_dict[row['Feature']] = (row['Threshold'], row['Direction'])
    
    # AND rule 평가
    result = evaluate_and_rule(df, category, thresholds_dict)
    and_rule_results.append(result)
    
    # 상세 출력
    print(f"\n[{category}]")
    print(f"  Rule 조건:")
    for feat, (thresh, direction) in thresholds_dict.items():
        op = ">=" if direction == 'high' else "<="
        print(f"    {feat} {op} {thresh:.4f}")
    print(f"\n  성능:")
    print(f"    TP={result['TP']:,} / {result['Attack_Total']:,} (Recall={result['Recall']:.4f})")
    print(f"    FP={result['FP']:,} / {result['Benign_Total']:,} (FPR={result['FPR']:.4f})")
    print(f"    Precision={result['Precision']:.4f}, F1={result['F1']:.4f}")

# 전체 요약 테이블
and_results_df = pd.DataFrame(and_rule_results)

print("\n" + "="*70)
print("AND Rule 성능 요약 테이블")
print("="*70)
print(f"{'Category':<15} {'TP':>8} {'FP':>8} {'FN':>8} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FPR':>10}")
print("-"*85)
for _, row in and_results_df.iterrows():
    print(f"{row['Category']:<15} {row['TP']:>8,} {row['FP']:>8,} {row['FN']:>8,} {row['Precision']:>10.4f} {row['Recall']:>10.4f} {row['F1']:>10.4f} {row['FPR']:>10.4f}")

# 문제 진단
print("\n" + "="*70)
print("진단: Recall 하락 원인 분석")
print("="*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    cat_df = results_df[results_df['Category'] == category]
    and_result = and_results_df[and_results_df['Category'] == category].iloc[0]
    
    single_recalls = cat_df['Recall'].values
    and_recall = and_result['Recall']
    
    # 단일 feature들의 recall 곱 (이론적 AND 최소)
    theoretical_min = np.prod(single_recalls)
    
    print(f"\n[{category}]")
    print(f"  단일 Feature Recall: {', '.join([f'{r:.4f}' for r in single_recalls])}")
    print(f"  이론적 최소 (곱): {theoretical_min:.4f}")
    print(f"  실제 AND Recall: {and_recall:.4f}")
    
    if and_recall < min(single_recalls) * 0.8:
        print(f"  ⚠️ 경고: AND 조합으로 Recall이 크게 하락함!")




Step 2: AND Rule 조합 성능 평가 (One-vs-Benign)

[Brute Force]
  Rule 조건:
    Bwd Pkts/s >= 55555.5556
    Init Fwd Win Byts >= 26883.0000

  성능:
    TP=2,170 / 2,324 (Recall=0.9337)
    FP=59 / 11,307 (FPR=0.0052)
    Precision=0.9735, F1=0.9532

[Botnet]
  Rule 조건:
    Flow Duration <= 11204.0000
    Pkt Len Mean <= 56.8750

  성능:
    TP=1,259 / 1,327 (Recall=0.9488)
    FP=2,461 / 11,307 (FPR=0.2177)
    Precision=0.3384, F1=0.4989

[DoS/DDoS]
  Rule 조건:
    PSH Flag Cnt >= 1.0000
    Init Fwd Win Byts >= 8192.0000

  성능:
    TP=7,033 / 8,145 (Recall=0.8635)
    FP=3,775 / 11,307 (FPR=0.3339)
    Precision=0.6507, F1=0.7422

[Web Attack]
  Rule 조건:
    Subflow Fwd Byts >= 3847.7000
    Fwd Header Len >= 811.2800
    Fwd Pkt Len Std >= 235.6157

  성능:
    TP=190 / 743 (Recall=0.2557)
    FP=17 / 11,307 (FPR=0.0015)
    Precision=0.9179, F1=0.4000

AND Rule 성능 요약 테이블
Category              TP       FP       FN  Precision     Recall         F1        FPR
------------------------------------

In [37]:
# Step 3: FPR 제한 기반 최적화
# ============================================================================

print("\n" + "="*70)
print("Step 3: FPR 제한 기반 Threshold 최적화")
print("="*70)

optimized_results = {}

for category, features in rule_config.items():
    alpha = fpr_limits[category]
    print(f"\n{'='*70}")
    print(f"[{category}] FPR 제한: α ≤ {alpha}")
    print("="*70)
    
    # 각 feature별 후보 생성
    feature_candidates = {}
    for feature, direction in features.items():
        candidates = get_threshold_candidates(df, category, feature, direction)
        feature_candidates[feature] = {
            'candidates': candidates,
            'direction': direction
        }
        print(f"\n{feature} ({direction}): {len(candidates)}개 후보")
        # 후보값 일부 출력
        sorted_vals = sorted(candidates.items(), key=lambda x: x[1])
        for name, val in sorted_vals[:3]:
            print(f"  {name}: {val:.4f}")
        if len(sorted_vals) > 3:
            print(f"  ... ({len(sorted_vals)-3}개 더)")
    
    # 모든 조합 탐색
    feature_names = list(features.keys())
    all_candidate_lists = []
    for feat in feature_names:
        cands = feature_candidates[feat]['candidates']
        all_candidate_lists.append([(name, val) for name, val in cands.items()])
    
    # 조합 수 계산
    total_combinations = 1
    for lst in all_candidate_lists:
        total_combinations *= len(lst)
    print(f"\n총 조합 수: {total_combinations}")
    
    # 탐색
    valid_candidates = []  # FPR ≤ α 만족하는 후보들
    
    for combo in product(*all_candidate_lists):
        # thresholds_dict 구성
        thresholds_dict = {}
        combo_info = {}
        for i, feat in enumerate(feature_names):
            name, val = combo[i]
            direction = feature_candidates[feat]['direction']
            thresholds_dict[feat] = (val, direction)
            combo_info[feat] = {'name': name, 'value': val}
        
        # AND rule 평가
        result = evaluate_and_rule(df, category, thresholds_dict)
        
        # 제약조건 체크: FPR ≤ α
        if result['FPR'] <= alpha:
            valid_candidates.append({
                'combo_info': combo_info,
                'thresholds_dict': thresholds_dict,
                'result': result
            })
    
    print(f"FPR ≤ {alpha} 만족하는 후보: {len(valid_candidates)}개")
    
    if len(valid_candidates) == 0:
        print(f"⚠️ 경고: FPR 제한을 만족하는 후보가 없음!")
        print(f"   → α를 높이거나 feature 재검토 필요")
        optimized_results[category] = None
        continue
    
    # 목적함수: Recall 최대화
    best = max(valid_candidates, key=lambda x: x['result']['Recall'])
    optimized_results[category] = best
    
    print(f"\n★ 최적 조합 (Recall 최대):")
    for feat, info in best['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best['result']
    print(f"\n  성능:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")


# ============================================================================
# 최종 요약
# ============================================================================

print("\n\n" + "="*70)
print("Step 3 최종 요약: 최적화된 Rule")
print("="*70)

summary_rows = []

for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    alpha = fpr_limits[category]
    best = optimized_results.get(category)
    
    print(f"\n[{category}] (α ≤ {alpha})")
    
    if best is None:
        print("  ❌ 유효한 후보 없음")
        continue
    
    features_info = rule_config[category]
    print("  IF")
    conditions = []
    for feat, direction in features_info.items():
        val = best['combo_info'][feat]['value']
        op = ">=" if direction == 'high' else "<="
        print(f"      {feat} {op} {val:.4f}")
        conditions.append(f"{feat} {op} {val:.4f}")
    print(f"  THEN \"{category}\"")
    
    r = best['result']
    summary_rows.append({
        'Category': category,
        'FPR_Limit': alpha,
        'Recall': r['Recall'],
        'FPR': r['FPR'],
        'Precision': r['Precision'],
        'F1': r['F1'],
        'TP': r['TP'],
        'FP': r['FP']
    })

# 요약 테이블
print("\n" + "="*70)
print("성능 요약 테이블")
print("="*70)
summary_df = pd.DataFrame(summary_rows)
print(f"{'Category':<15} {'α':>6} {'Recall':>10} {'FPR':>10} {'Precision':>10} {'F1':>10}")
print("-"*65)
for _, row in summary_df.iterrows():
    print(f"{row['Category']:<15} {row['FPR_Limit']:>6.2f} {row['Recall']:>10.4f} {row['FPR']:>10.4f} {row['Precision']:>10.4f} {row['F1']:>10.4f}")


# ============================================================================
# Step 2 대비 개선 비교
# ============================================================================

print("\n" + "="*70)
print("Step 2 대비 개선 비교")
print("="*70)

# Step 2 결과 (이전 출력에서 가져옴)
step2_results = {
    'Brute Force': {'Recall': 0.9337, 'FPR': 0.0052, 'F1': 0.9532},
    'Botnet': {'Recall': 0.9488, 'FPR': 0.2177, 'F1': 0.4989},
    'DoS/DDoS': {'Recall': 0.8134, 'FPR': 0.2150, 'F1': 0.7703},
    'Web Attack': {'Recall': 0.2557, 'FPR': 0.0015, 'F1': 0.4000}
}

print(f"{'Category':<15} {'Step2 FPR':>12} {'Step3 FPR':>12} {'Step2 Recall':>14} {'Step3 Recall':>14}")
print("-"*70)
for category in ['Brute Force', 'Botnet', 'DoS/DDoS', 'Web Attack']:
    s2 = step2_results[category]
    best = optimized_results.get(category)
    if best:
        s3 = best['result']
        fpr_change = "↓" if s3['FPR'] < s2['FPR'] else ("↑" if s3['FPR'] > s2['FPR'] else "=")
        recall_change = "↓" if s3['Recall'] < s2['Recall'] else ("↑" if s3['Recall'] > s2['Recall'] else "=")
        print(f"{category:<15} {s2['FPR']:>10.4f}   {s3['FPR']:>10.4f} {fpr_change}  {s2['Recall']:>12.4f}   {s3['Recall']:>12.4f} {recall_change}")
    else:
        print(f"{category:<15} {s2['FPR']:>10.4f}   {'N/A':>10}    {s2['Recall']:>12.4f}   {'N/A':>12}")



Step 3: FPR 제한 기반 Threshold 최적화

[Brute Force] FPR 제한: α ≤ 0.01

Bwd Pkts/s (high): 9개 후보
  benign_p90: 621.2257
  benign_p92: 2808.9888
  attack_p1: 38815.3846
  ... (6개 더)

Init Fwd Win Byts (high): 5개 후보
  benign_p90: 14600.0000
  benign_p92: 26883.0000
  benign_p95: 32818.1000
  ... (2개 더)

총 조합 수: 45
FPR ≤ 0.01 만족하는 후보: 27개

★ 최적 조합 (Recall 최대):
  Bwd Pkts/s >= 51754.3860 (benign_p97)
  Init Fwd Win Byts >= 26883.0000 (benign_p92)

  성능:
    Recall=0.9660, FPR=0.0088
    Precision=0.9578, F1=0.9619
    TP=2245, FP=99, FN=79

[Botnet] FPR 제한: α ≤ 0.05

Flow Duration (low): 11개 후보
  benign_p1: 1.0000
  benign_p2: 18.0000
  benign_p5: 20.0000
  ... (8개 더)

Pkt Len Mean (low): 2개 후보
  benign_p10: 0.0000
  attack_p99: 56.8750

총 조합 수: 22
FPR ≤ 0.05 만족하는 후보: 5개

★ 최적 조합 (Recall 최대):
  Flow Duration <= 20.0000 (benign_p5)
  Pkt Len Mean <= 0.0000 (benign_p10)

  성능:
    Recall=0.0000, FPR=0.0440
    Precision=0.0000, F1=0.0000
    TP=0, FP=497, FN=1327

[DoS/DDoS] FPR 제한: α ≤ 0.05

PSH 

In [38]:
# Step 4A: Web Attack k-of-n (2-of-3)
# ============================================================================

print("\n" + "="*70)
print("Step 4A: Web Attack k-of-n (k=2, n=3)")
print("="*70)

category = 'Web Attack'
config = experiment_config[category]
features = rule_config[category]
k = config['k']
alpha = config['alpha']

print(f"설정: k={k} (3개 조건 중 {k}개 이상 만족)")
print(f"FPR 제한: α ≤ {alpha}")

# 후보 생성
feature_candidates = {}
for feature, direction in features.items():
    candidates = get_threshold_candidates(df, category, feature, direction)
    feature_candidates[feature] = {
        'candidates': candidates,
        'direction': direction
    }
    print(f"\n{feature} ({direction}): {len(candidates)}개 후보")

# 모든 조합 탐색
feature_names = list(features.keys())
all_candidate_lists = []
for feat in feature_names:
    cands = feature_candidates[feat]['candidates']
    all_candidate_lists.append([(name, val) for name, val in cands.items()])

total_combinations = 1
for lst in all_candidate_lists:
    total_combinations *= len(lst)
print(f"\n총 조합 수: {total_combinations}")

# k-of-n 탐색
valid_candidates = []

for combo in product(*all_candidate_lists):
    thresholds_dict = {}
    combo_info = {}
    for i, feat in enumerate(feature_names):
        name, val = combo[i]
        direction = feature_candidates[feat]['direction']
        thresholds_dict[feat] = (val, direction)
        combo_info[feat] = {'name': name, 'value': val}
    
    # k-of-n 평가
    result = evaluate_k_of_n_rule(df, category, thresholds_dict, k)
    
    if result['FPR'] <= alpha:
        valid_candidates.append({
            'combo_info': combo_info,
            'thresholds_dict': thresholds_dict,
            'result': result
        })

print(f"FPR ≤ {alpha} 만족하는 후보: {len(valid_candidates)}개")

if len(valid_candidates) > 0:
    # Recall 최대화
    best_web = max(valid_candidates, key=lambda x: x['result']['Recall'])
    
    print(f"\n★ 최적 조합 (k={k}, Recall 최대):")
    for feat, info in best_web['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best_web['result']
    print(f"\n  성능:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")
    
    # Step 3 AND rule 대비 비교
    print(f"\n  Step 3 (AND) 대비:")
    print(f"    Recall: 0.2638 → {r['Recall']:.4f} ({'+' if r['Recall'] > 0.2638 else ''}{(r['Recall']-0.2638)*100:.1f}%p)")
    print(f"    FPR:    0.0096 → {r['FPR']:.4f}")
else:
    best_web = None
    print("⚠️ 유효한 후보 없음")

# ============================================================================
# Step 4B: DoS/DDoS alpha = 0.10
# ============================================================================

print("\n\n" + "="*70)
print("Step 4B: DoS/DDoS α = 0.10 (완화)")
print("="*70)

category = 'DoS/DDoS'
config = experiment_config[category]
features = rule_config[category]
alpha = config['alpha']

print(f"FPR 제한: α ≤ {alpha} (기존 0.05에서 완화)")

# 후보 생성
feature_candidates = {}
for feature, direction in features.items():
    candidates = get_threshold_candidates(df, category, feature, direction)
    feature_candidates[feature] = {
        'candidates': candidates,
        'direction': direction
    }
    print(f"\n{feature} ({direction}): {len(candidates)}개 후보")

# 모든 조합 탐색
feature_names = list(features.keys())
all_candidate_lists = []
for feat in feature_names:
    cands = feature_candidates[feat]['candidates']
    all_candidate_lists.append([(name, val) for name, val in cands.items()])

total_combinations = 1
for lst in all_candidate_lists:
    total_combinations *= len(lst)
print(f"\n총 조합 수: {total_combinations}")

valid_candidates = []

for combo in product(*all_candidate_lists):
    thresholds_dict = {}
    combo_info = {}
    for i, feat in enumerate(feature_names):
        name, val = combo[i]
        direction = feature_candidates[feat]['direction']
        thresholds_dict[feat] = (val, direction)
        combo_info[feat] = {'name': name, 'value': val}
    
    result = evaluate_and_rule(df, category, thresholds_dict)
    
    if result['FPR'] <= alpha:
        valid_candidates.append({
            'combo_info': combo_info,
            'thresholds_dict': thresholds_dict,
            'result': result
        })

print(f"FPR ≤ {alpha} 만족하는 후보: {len(valid_candidates)}개")

if len(valid_candidates) > 0:
    best_dos = max(valid_candidates, key=lambda x: x['result']['Recall'])
    
    print(f"\n★ 최적 조합 (Recall 최대):")
    for feat, info in best_dos['combo_info'].items():
        direction = feature_candidates[feat]['direction']
        op = ">=" if direction == 'high' else "<="
        print(f"  {feat} {op} {info['value']:.4f} ({info['name']})")
    
    r = best_dos['result']
    print(f"\n  성능:")
    print(f"    Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}")
    print(f"    Precision={r['Precision']:.4f}, F1={r['F1']:.4f}")
    print(f"    TP={r['TP']}, FP={r['FP']}, FN={r['FN']}")
    
    # Step 3 대비 비교
    print(f"\n  Step 3 (α=0.05) 대비:")
    print(f"    Recall: 0.5650 → {r['Recall']:.4f} ({'+' if r['Recall'] > 0.5650 else ''}{(r['Recall']-0.5650)*100:.1f}%p)")
    print(f"    FPR:    0.0438 → {r['FPR']:.4f}")
else:
    best_dos = None
    print("⚠️ 유효한 후보 없음")




Step 4A: Web Attack k-of-n (k=2, n=3)
설정: k=2 (3개 조건 중 2개 이상 만족)
FPR 제한: α ≤ 0.01

Subflow Fwd Byts (high): 3개 후보

Fwd Header Len (high): 7개 후보

Fwd Pkt Len Std (high): 6개 후보

총 조합 수: 126
FPR ≤ 0.01 만족하는 후보: 22개

★ 최적 조합 (k=2, Recall 최대):
  Subflow Fwd Byts >= 24782.0440 (benign_p99)
  Fwd Header Len >= 400.0000 (benign_p95)
  Fwd Pkt Len Std >= 235.6157 (benign_p95)

  성능:
    Recall=0.2611, FPR=0.0082
    Precision=0.6760, F1=0.3767
    TP=194, FP=93, FN=549

  Step 3 (AND) 대비:
    Recall: 0.2638 → 0.2611 (-0.3%p)
    FPR:    0.0096 → 0.0082


Step 4B: DoS/DDoS α = 0.10 (완화)
FPR 제한: α ≤ 0.1 (기존 0.05에서 완화)

PSH Flag Cnt (high): 2개 후보

Init Fwd Win Byts (high): 6개 후보

총 조합 수: 12
FPR ≤ 0.1 만족하는 후보: 10개

★ 최적 조합 (Recall 최대):
  PSH Flag Cnt >= 1.0000 (benign_p90)
  Init Fwd Win Byts >= 14600.0000 (benign_p90)

  성능:
    Recall=0.5751, FPR=0.0562
    Precision=0.8806, F1=0.6958
    TP=4684, FP=635, FN=3461

  Step 3 (α=0.05) 대비:
    Recall: 0.5650 → 0.5751 (+1.0%p)
    FPR:    0.0438 → 0.

In [39]:
# Step 4 최종 요약
# ============================================================================

print("\n\n" + "="*70)
print("Step 4 최종 요약")
print("="*70)

# Step 3 결과 (비교용)
step3_results = {
    'Brute Force': {'Recall': 0.9660, 'FPR': 0.0088, 'Precision': 0.9578, 'F1': 0.9619},
    'Botnet': {'Recall': 0.0000, 'FPR': 0.0440, 'Precision': 0.0000, 'F1': 0.0000},
    'DoS/DDoS': {'Recall': 0.5650, 'FPR': 0.0438, 'Precision': 0.9029, 'F1': 0.6951},
    'Web Attack': {'Recall': 0.2638, 'FPR': 0.0096, 'Precision': 0.6426, 'F1': 0.3740}
}

print("\n[카테고리별 최종 Rule]")
print("-"*70)

# Brute Force (Step 3 유지)
print("\n1. Brute Force (Step 3 유지, α=0.01)")
print("   IF Bwd Pkts/s >= 51754.3860")
print("      AND Init Fwd Win Byts >= 26883.0000")
print("   THEN \"Brute Force\"")
s3 = step3_results['Brute Force']
print(f"   → Recall={s3['Recall']:.4f}, FPR={s3['FPR']:.4f}, F1={s3['F1']:.4f}")

# Botnet (실패 케이스)
print("\n2. Botnet (Step 3, α=0.05) ❌ 실패")
print("   IF Flow Duration <= 20.0000")
print("      AND Pkt Len Mean <= 0.0000")
print("   THEN \"Botnet\"")
print("   → Recall=0.0000, FPR=0.0440 (Feature 한계로 탐지 불가)")

# DoS/DDoS (Step 4B)
print("\n3. DoS/DDoS (Step 4B, α=0.10)")
if best_dos:
    for feat, info in best_dos['combo_info'].items():
        direction = rule_config['DoS/DDoS'][feat]
        op = ">=" if direction == 'high' else "<="
        print(f"   IF {feat} {op} {info['value']:.4f}")
    print("   THEN \"DoS/DDoS\"")
    r = best_dos['result']
    print(f"   → Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}, F1={r['F1']:.4f}")

# Web Attack (Step 4A)
print("\n4. Web Attack (Step 4A, k=2-of-3, α=0.01)")
if best_web:
    print("   IF 2개 이상 만족:")
    for feat, info in best_web['combo_info'].items():
        direction = rule_config['Web Attack'][feat]
        op = ">=" if direction == 'high' else "<="
        print(f"      - {feat} {op} {info['value']:.4f}")
    print("   THEN \"Web Attack 의심\"")
    r = best_web['result']
    print(f"   → Recall={r['Recall']:.4f}, FPR={r['FPR']:.4f}, F1={r['F1']:.4f}")


# 성능 비교 테이블
print("\n" + "="*70)
print("Step 3 vs Step 4 성능 비교")
print("="*70)
print(f"{'Category':<15} {'Step':>8} {'Recall':>10} {'FPR':>10} {'F1':>10} {'비고':<20}")
print("-"*75)

# Brute Force
s3 = step3_results['Brute Force']
print(f"{'Brute Force':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'유지':<20}")

# Botnet
print(f"{'Botnet':<15} {'Step3':>8} {'0.0000':>10} {'0.0440':>10} {'0.0000':>10} {'실패 (발표용)':<20}")

# DoS/DDoS
s3 = step3_results['DoS/DDoS']
print(f"{'DoS/DDoS':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'α=0.05':<20}")
if best_dos:
    r = best_dos['result']
    change = "↑" if r['Recall'] > s3['Recall'] else "↓"
    print(f"{'':<15} {'Step4B':>8} {r['Recall']:>10.4f} {r['FPR']:>10.4f} {r['F1']:>10.4f} {'α=0.10 ' + change:<20}")

# Web Attack
s3 = step3_results['Web Attack']
print(f"{'Web Attack':<15} {'Step3':>8} {s3['Recall']:>10.4f} {s3['FPR']:>10.4f} {s3['F1']:>10.4f} {'AND rule':<20}")
if best_web:
    r = best_web['result']
    change = "↑" if r['Recall'] > s3['Recall'] else "↓"
    print(f"{'':<15} {'Step4A':>8} {r['Recall']:>10.4f} {r['FPR']:>10.4f} {r['F1']:>10.4f} {'k=2-of-3 ' + change:<20}")



Step 4 최종 요약

[카테고리별 최종 Rule]
----------------------------------------------------------------------

1. Brute Force (Step 3 유지, α=0.01)
   IF Bwd Pkts/s >= 51754.3860
      AND Init Fwd Win Byts >= 26883.0000
   THEN "Brute Force"
   → Recall=0.9660, FPR=0.0088, F1=0.9619

2. Botnet (Step 3, α=0.05) ❌ 실패
   IF Flow Duration <= 20.0000
      AND Pkt Len Mean <= 0.0000
   THEN "Botnet"
   → Recall=0.0000, FPR=0.0440 (Feature 한계로 탐지 불가)

3. DoS/DDoS (Step 4B, α=0.10)
   IF PSH Flag Cnt >= 1.0000
   IF Init Fwd Win Byts >= 14600.0000
   THEN "DoS/DDoS"
   → Recall=0.5751, FPR=0.0562, F1=0.6958

4. Web Attack (Step 4A, k=2-of-3, α=0.01)
   IF 2개 이상 만족:
      - Subflow Fwd Byts >= 24782.0440
      - Fwd Header Len >= 400.0000
      - Fwd Pkt Len Std >= 235.6157
   THEN "Web Attack 의심"
   → Recall=0.2611, FPR=0.0082, F1=0.3767

Step 3 vs Step 4 성능 비교
Category            Step     Recall        FPR         F1 비고                  
--------------------------------------------------------------